# 04 — Model Eğitimi

**Deney Protokolü:**
- **Deney 1:** Tam özellik seti (tüm modeller)
- **Deney 2:** Vajrobol'un 5 özelliği (özellikle LR)
- **Deney 3:** RF Top-20 özellik (tüm modeller)
- **Deney 4:** 10-Fold Cross Validation (en iyi modeller)

**Not:** SVM tam veri setinde yavaş olabilir. İlk denemede alt örneklem (50K) ile çalıştırılabilir.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from src.preprocessing import full_preprocessing_pipeline
from src.feature_selection import get_top5_vajrobol
from src.models import get_baseline_models, train_model
from src.evaluation import evaluate_model, compare_models_table, cross_validate_model

DATA_PATH = '../data/PhiUSIIL_Phishing_URL_Dataset.csv'
METRICS_DIR = '../results/metrics/'
FIG_DIR = '../results/figures/'
FIG_KWARGS = dict(dpi=150, bbox_inches='tight')

os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

sns.set_theme(style='whitegrid')

## 1. Veri Hazırlığı

In [ ]:
X_train, X_test, y_train, y_test, tld_encoder, scaler = full_preprocessing_pipeline(DATA_PATH)
feature_names = list(X_train.columns)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 2. RF Top-20 Özelliklerini Belirle

In [ ]:
# Kaydedilmiş özellik setlerini yükle (03_feature_selection.ipynb çalıştırıldıysa)
feature_sets_path = f'{METRICS_DIR}feature_sets.joblib'

if os.path.exists(feature_sets_path):
    feature_sets = joblib.load(feature_sets_path)
    rf_top20 = feature_sets['rf_top20']
    vajrobol_5 = feature_sets['vajrobol_5']
    print('Kaydedilmiş özellik setleri yüklendi.')
else:
    # Anlık hesapla
    from src.feature_selection import rf_feature_importance
    print('RF önem skorları hesaplanıyor...')
    rf_top20, _ = rf_feature_importance(X_train, y_train, top_n=20)
    vajrobol_5 = [f for f in get_top5_vajrobol() if f in feature_names]

print(f'RF Top-20: {rf_top20}')
print(f'Vajrobol-5: {vajrobol_5}')

---
## DENEY 1: Tam Özellik Seti

In [ ]:
print('=== DENEY 1: Tam Özellik Seti ===\n')
models_exp1 = get_baseline_models()
results_exp1 = {}
trained_models_exp1 = {}

for name, model in models_exp1.items():
    print(f'[{name}] eğitiliyor...')
    model, elapsed = train_model(model, X_train, y_train)
    metrics = evaluate_model(model, X_test, y_test)
    metrics['Train Time (s)'] = round(elapsed, 2)
    results_exp1[name] = metrics
    trained_models_exp1[name] = model
    print(f'  Acc={metrics["Accuracy"]:.4f} | F1={metrics["F1-Score"]:.4f} | Recall={metrics["Recall"]:.4f} | AUC={metrics["ROC-AUC"]:.4f}\n')

df_exp1 = compare_models_table(results_exp1)
print('\n=== DENEY 1 SONUÇLARI ===')
print(df_exp1.to_string())

df_exp1.to_csv(f'{METRICS_DIR}exp1_full_features.csv')
joblib.dump(trained_models_exp1, f'{METRICS_DIR}trained_models_exp1.joblib')

---
## DENEY 2: Vajrobol'un 5 Özelliği

In [ ]:
print('=== DENEY 2: Vajrobol 5 Özellik ===\n')

X_train_v5 = X_train[vajrobol_5]
X_test_v5 = X_test[vajrobol_5]

models_exp2 = get_baseline_models()
results_exp2 = {}
trained_models_exp2 = {}

for name, model in models_exp2.items():
    print(f'[{name}] eğitiliyor (5 özellik)...')
    model, elapsed = train_model(model, X_train_v5, y_train)
    metrics = evaluate_model(model, X_test_v5, y_test)
    metrics['Train Time (s)'] = round(elapsed, 2)
    results_exp2[name] = metrics
    trained_models_exp2[name] = model
    print(f'  Acc={metrics["Accuracy"]:.4f} | F1={metrics["F1-Score"]:.4f} | Recall={metrics["Recall"]:.4f}\n')

df_exp2 = compare_models_table(results_exp2)
print('\n=== DENEY 2 SONUÇLARI ===')
print(df_exp2.to_string())
print(f'\nHedef: Vajrobol LR Accuracy ~%99.97 → LR: {results_exp2["Logistic Regression"]["Accuracy"]:.4f}')

df_exp2.to_csv(f'{METRICS_DIR}exp2_vajrobol5.csv')
joblib.dump(trained_models_exp2, f'{METRICS_DIR}trained_models_exp2.joblib')

---
## DENEY 3: RF Top-20 Özellik

In [ ]:
print('=== DENEY 3: RF Top-20 Özellik ===\n')

X_train_rf20 = X_train[rf_top20]
X_test_rf20 = X_test[rf_top20]

models_exp3 = get_baseline_models()
results_exp3 = {}
trained_models_exp3 = {}

for name, model in models_exp3.items():
    print(f'[{name}] eğitiliyor (RF Top-20)...')
    model, elapsed = train_model(model, X_train_rf20, y_train)
    metrics = evaluate_model(model, X_test_rf20, y_test)
    metrics['Train Time (s)'] = round(elapsed, 2)
    results_exp3[name] = metrics
    trained_models_exp3[name] = model
    print(f'  Acc={metrics["Accuracy"]:.4f} | F1={metrics["F1-Score"]:.4f} | Recall={metrics["Recall"]:.4f}\n')

df_exp3 = compare_models_table(results_exp3)
print('\n=== DENEY 3 SONUÇLARI ===')
print(df_exp3.to_string())

df_exp3.to_csv(f'{METRICS_DIR}exp3_rf_top20.csv')
joblib.dump(trained_models_exp3, f'{METRICS_DIR}trained_models_exp3.joblib')

---
## DENEY 4: 10-Fold Cross Validation (En iyi 3 model)

In [ ]:
print('=== DENEY 4: 10-Fold Cross Validation ===\n')

# Tam veri üzerinde CV (train + test birleşik)
X_full = pd.concat([X_train, X_test])
y_full = pd.concat([y_train, y_test])

# En iyi 3 modeli seç (Deney 1 sonuçlarına göre)
best3_names = df_exp1.head(3).index.tolist()
print(f'CV için seçilen modeller: {best3_names}')

cv_results = {}
for name in best3_names:
    print(f'\n[{name}] 10-Fold CV...')
    model = get_baseline_models()[name]
    cv_results[name] = cross_validate_model(model, X_full, y_full, cv=10)

# CV sonuçlarını DataFrame'e dönüştür
cv_summary = []
for model_name, metrics in cv_results.items():
    row = {'Model': model_name}
    for metric, vals in metrics.items():
        row[f'{metric}_mean'] = round(vals['mean'], 4)
        row[f'{metric}_std'] = round(vals['std'], 4)
    cv_summary.append(row)

df_cv = pd.DataFrame(cv_summary).set_index('Model')
print('\n=== DENEY 4 CV SONUÇLARI ===')
print(df_cv.to_string())

df_cv.to_csv(f'{METRICS_DIR}exp4_cross_validation.csv')
print('\nCV sonuçları kaydedildi.')